
# S8 - Procesamiento en streaming con Spark sobre `red-telemetria` (Proyecto Sello)

Actividad autónoma de S8 (sección 4.1) sobre la fuente real de mi Proyecto Sello: el topic `red-telemetria` de S7, que
alimenta en vivo el agente `uso-red/agente/agente_red.py` con los contadores reales de las tarjetas de red de esta PC
(contrato `red.lectura`, key = `sensorId`, 3 particiones).

Antes de ejecutar: Kafka, `uso-red` (bridge) y el agente deben estar corriendo, y `pyspark` debe levantarse con
`docker compose -f compose.yml -f compose.kafka.yml up -d`.

| Tarea 4.1 | Sección |
|---|---|
| 1. Lectura streaming con esquema explícito | 1-2 |
| 2. Ventana con watermarking, margen justificado | 3-4 |
| 3. Dato tardío descartado (y arranque en frío) | 5 |
| 4. Duplicado filtrado por `dropDuplicates` + watermark | 6 |
| 5. Dos intervalos de disparo y su `batchId` | 7 |
| 6. Parquet con checkpoint, dos corridas que acumulan | 8 |


## 0. SparkSession con el conector de Kafka

In [1]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s08-streaming-red-telemetria")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    # Las ventanas se muestran en hora de Perú, la misma del reloj de Windows del agente.
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c90203b0-48bd-4243-acb7-78fae3f6743f;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central


	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 260ms :: artifacts dl 8ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central i

26/10/08 22:08:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


In [2]:

import json, os, shutil, threading, time
from datetime import datetime, timedelta, timezone
from pyspark.sql.functions import (avg, col, count, from_json, greatest, lit, max as smax,
                                   round as sround, to_json, struct, when, window)
from pyspark.sql.types import DoubleType, LongType, StringType, StructType

BOOTSTRAP = "kafka:9092"
TOPIC = "red-telemetria"
BASE = "./artifacts"

LIMA = timezone(timedelta(hours=-5))   # hora de Perú (sin horario de verano), igual que spark.sql.session.timeZone

def hora(ms):
    return datetime.fromtimestamp(ms / 1000, LIMA).strftime("%H:%M:%S")

def ventana_de(ms, ancho=10):
    # Ventana fija de `ancho` s a la que pertenece un tiempo de evento, como texto HH:MM:SS-HH:MM:SS.
    inicio = (ms // 1000) // ancho * ancho
    return f"{hora(inicio * 1000)}-{hora((inicio + ancho) * 1000)}"

def wm_lima(iso):
    # Watermark de lastProgress (ISO UTC) en hora de Lima.
    if not iso or iso.startswith("1970"):
        return "sin referencia (arranque)"
    return datetime.fromisoformat(iso.replace("Z", "+00:00")).astimezone(LIMA).strftime("%H:%M:%S")

def limpiar(ruta):
    shutil.rmtree(ruta, ignore_errors=True)

# Lectura batch: el topic ya tiene datos reales de S7 y de hoy.
lote = (spark.read.format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP)
        .option("subscribe", TOPIC)
        .option("startingOffsets", "earliest")
        .load())
print("mensajes en el topic:", lote.count())
(lote.selectExpr("CAST(key AS STRING) AS key", "partition", "offset", "CAST(value AS STRING) AS value")
     .orderBy(col("timestamp").desc()).show(4, truncate=90))


mensajes en el topic: 2150


+-----------------------------------+---------+------+------------------------------------------------------------------------------------------+
|                                key|partition|offset|                                                                                     value|
+-----------------------------------+---------+------+------------------------------------------------------------------------------------------+
|                     nsmj-tailscale|        0|  1079|{"tipoEvento": "red.lectura", "sensorId": "nsmj-tailscale", "host": "nsmj", "interfaz":...|
|nsmj-vethernet-wsl-hyper-v-firewall|        2|   536|{"tipoEvento": "red.lectura", "sensorId": "nsmj-vethernet-wsl-hyper-v-firewall", "host"...|
|                    nsmj-ethernet-2|        1|   536|{"tipoEvento": "red.lectura", "sensorId": "nsmj-ethernet-2", "host": "nsmj", "interfaz"...|
|                      nsmj-ethernet|        0|  1078|{"tipoEvento": "red.lectura", "sensorId": "nsmj-ethernet", "host": "ns

## 1. Lectura en modo streaming

In [3]:

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", BOOTSTRAP)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .load())
print("es un stream:", crudo.isStreaming)
crudo.printSchema()


es un stream: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## 2. Esquema explícito del contrato `red.lectura` (S7)

In [4]:

esquema_red = (
    StructType()
    .add("tipoEvento", StringType())
    .add("sensorId", StringType())
    .add("host", StringType())
    .add("interfaz", StringType())
    .add("tipoInterfaz", StringType())
    .add("velocidadEnlaceMbps", DoubleType())
    .add("intervaloMs", LongType())
    .add("bytesRxPorSeg", DoubleType())
    .add("bytesTxPorSeg", DoubleType())
    .add("paquetesRxPorSeg", DoubleType())
    .add("paquetesTxPorSeg", DoubleType())
    .add("erroresPorSeg", DoubleType())
    .add("descartesPorSeg", DoubleType())
    .add("origen", StringType())
    .add("timestamp", LongType())
)

def parsear(df):
    return (df.select(col("partition"), col("offset"), col("timestamp").alias("kafka_ts"),
                      from_json(col("value").cast("string"), esquema_red).alias("e"))
              .select("partition", "offset", "kafka_ts", "e.*")
              .filter(col("tipoEvento") == "red.lectura")
              .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
              .withColumn("usoEnlacePct",
                          when(col("velocidadEnlaceMbps") > 0,
                               greatest("bytesRxPorSeg", "bytesTxPorSeg") / (col("velocidadEnlaceMbps") * 125000) * 100)))

lecturas = parsear(crudo)
lecturas.printSchema()


root
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_ts: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- host: string (nullable = true)
 |-- interfaz: string (nullable = true)
 |-- tipoInterfaz: string (nullable = true)
 |-- velocidadEnlaceMbps: double (nullable = true)
 |-- intervaloMs: long (nullable = true)
 |-- bytesRxPorSeg: double (nullable = true)
 |-- bytesTxPorSeg: double (nullable = true)
 |-- paquetesRxPorSeg: double (nullable = true)
 |-- paquetesTxPorSeg: double (nullable = true)
 |-- erroresPorSeg: double (nullable = true)
 |-- descartesPorSeg: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- usoEnlacePct: double (nullable = true)




### Justificación del margen del watermark: latencia real del transporte

`ts` es el tiempo de evento (cuándo midió el agente). `kafka_ts` es cuándo el broker guardó el mensaje. Su diferencia es
cuánto tarda una lectura real en recorrer agente → MQTT → bridge → Kafka: es el retraso que el watermark tiene que tolerar.


In [5]:

from pyspark.sql.functions import expr, percentile_approx

historico = parsear(lote).filter(col("origen") == "agente-red")
retraso = historico.select(((col("kafka_ts").cast("double") * 1000) - col("timestamp")).alias("retraso_ms"))
(retraso.agg(count("*").alias("lecturas"),
             percentile_approx("retraso_ms", 0.5).alias("p50_ms"),
             percentile_approx("retraso_ms", 0.99).alias("p99_ms"),
             smax("retraso_ms").alias("max_ms"))
        .show())


+--------+------+------+------+
|lecturas|p50_ms|p99_ms|max_ms|
+--------+------+------+------+
|    2152|   3.0|   9.0|  42.0|
+--------+------+------+------+




## 3. Agregación por ventana fija de 10 s con watermark de 10 s

Métrica del dominio (mi dimensión U2, volumen de tráfico): por sensor y por ventana, el promedio y el máximo de bytes
recibidos por segundo y el uso máximo del enlace. Cada micro-lote se recolecta con `foreachBatch` para ver la salida
dentro del notebook (el *sink* `console` imprime en el log del contenedor, no aquí).


In [6]:

def correr(df, nombre, segundos, modo="update", trigger="5 seconds", chk=None, mientras=None):
    """Corre una consulta un tiempo fijo de reloj y devuelve (lotes, progreso)."""
    lotes = []
    def recolectar(batch_df, batch_id):
        lotes.append((batch_id, batch_df.toPandas()))
    escritor = (df.writeStream.outputMode(modo).foreachBatch(recolectar)
                .trigger(processingTime=trigger).queryName(nombre))
    if chk:
        escritor = escritor.option("checkpointLocation", chk)
    q = escritor.start()
    t0 = time.time()
    if mientras:
        threading.Thread(target=mientras, args=(q, t0), daemon=True).start()
    q.awaitTermination(segundos)
    progreso = q.recentProgress
    q.stop()
    return lotes, progreso

def marca_de_agua(progreso):
    return {p["batchId"]: p.get("eventTime", {}).get("watermark") for p in progreso}

def mostrar(lotes, progreso, columnas=None, max_lotes=None):
    wm = marca_de_agua(progreso)
    for batch_id, pdf in lotes[:max_lotes]:
        print(f"--- Batch {batch_id}  (watermark: {wm_lima(wm.get(batch_id))}) ---")
        print("(sin filas nuevas)" if pdf.empty else (pdf[columnas] if columnas else pdf).to_string(index=False))

def ventana_txt(pdf):
    pdf = pdf.copy()
    pdf["ventana"] = pdf["window"].apply(lambda w: f"{w['start']:%H:%M:%S}-{w['end']:%H:%M:%S}")
    return pdf

CHK_VENTANA = f"{BASE}/chk_ventana_10s"
limpiar(CHK_VENTANA)

agregado = (lecturas
    .withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(count("*").alias("n"),
         sround(avg("bytesRxPorSeg") / 1000, 1).alias("rx_prom_KBs"),
         sround(smax("bytesRxPorSeg") / 1000, 1).alias("rx_max_KBs"),
         sround(smax("usoEnlacePct"), 2).alias("uso_max_pct")))

lotes_v, prog_v = correr(agregado, "ventana_10s", 45, chk=CHK_VENTANA)
lotes_v = [(b, ventana_txt(p) if not p.empty else p) for b, p in lotes_v]
mostrar(lotes_v, prog_v, ["ventana", "sensorId", "n", "rx_prom_KBs", "rx_max_KBs", "uso_max_pct"])


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


...

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


--- Batch 0  (watermark: sin referencia (arranque)) ---
(sin filas nuevas)
--- Batch 1  (watermark: sin referencia (arranque)) ---
          ventana                            sensorId  n  rx_prom_KBs  rx_max_KBs  uso_max_pct
17:08:30-17:08:40                      nsmj-tailscale  4          0.0         0.0         0.00
17:08:30-17:08:40                     nsmj-ethernet-2  4          0.0         0.0         0.00
17:08:30-17:08:40                       nsmj-ethernet  4          5.2         9.7         0.37
17:08:30-17:08:40 nsmj-vethernet-wsl-hyper-v-firewall  4          0.0         0.0         0.00
--- Batch 2  (watermark: 17:08:24) ---
          ventana                            sensorId  n  rx_prom_KBs  rx_max_KBs  uso_max_pct
17:08:30-17:08:40                      nsmj-tailscale  9          0.0         0.0         0.00
17:08:30-17:08:40                     nsmj-ethernet-2  9          0.0         0.0         0.00
17:08:30-17:08:40                       nsmj-ethernet  9          2.5 

### 4. Ventanas cerradas: ninguna vuelve a aparecer después de que el watermark pasa su fi

In [7]:

apariciones = {}
for b, p in lotes_v:
    for v in (p["ventana"].unique() if not p.empty else []):
        apariciones.setdefault(v, []).append(b)
wm = {b: wm_lima(w) for b, w in marca_de_agua(prog_v).items()}
print(f"{'ventana':<20}{'sale en batchId':<22}{'cerrada desde batch':<22}watermark de ese batch")
for v, bs in sorted(apariciones.items()):
    fin = v.split("-")[1]
    cierre = next((b for b, w in sorted(wm.items()) if w[:2].isdigit() and w >= fin), None)
    print(f"{v:<20}{str(bs):<22}{str(cierre):<22}{wm.get(cierre, 'todavía abierta')}")


ventana             sale en batchId       cerrada desde batch   watermark de ese batch
17:08:30-17:08:40   [1, 2]                6                     17:08:44
17:08:40-17:08:50   [3, 4]                8                     17:08:54
17:08:50-17:09:00   [5, 6]                None                  todavía abierta
17:09:00-17:09:10   [7, 8]                None                  todavía abierta
17:09:10-17:09:20   [9]                   None                  todavía abierta



## 5. Dato tardío real descartado por el watermark

Con la consulta corriendo, a los 35 s se toma una **lectura real** de `nsmj-ethernet` que el agente midió hace ~30 s y se
vuelve a publicar con su `timestamp` original, como lo haría el agente si hubiera guardado en buffer una lectura durante un
corte de MQTT. Solo cambia `origen` a `"reenvio-tardio"`, para poder rastrearla en Kafka. Si el watermark la aceptara, su
ventana (ya emitida y cerrada) volvería a salir con `n + 1`.


In [8]:

CHK_TARDIO = f"{BASE}/chk_tardio"
limpiar(CHK_TARDIO)
registro_tardio = {}

def leer_lectura_real(sensor, segundos_atras):
    """Lectura real del sensor medida hace aprox. `segundos_atras` (batch sobre el topic)."""
    objetivo = int(time.time() * 1000) - segundos_atras * 1000
    filas = (parsear(spark.read.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
                     .option("subscribe", TOPIC).option("startingOffsets", "earliest").load()
                     .filter(col("timestamp") >= expr(f"timestamp_millis({objetivo - 1500})"))
                     .filter(col("timestamp") <= expr(f"timestamp_millis({objetivo + 1500})"))
                     .withColumn("value_str", col("value").cast("string"))
                     .select("partition", "offset", "timestamp", "value"))
             .filter((col("sensorId") == sensor) & (col("origen") == "agente-red"))
             .orderBy("timestamp").limit(1).collect())
    return filas[0] if filas else None

def valor_kafka(sensor, ts):
    df = (spark.read.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
          .option("subscribe", TOPIC).option("startingOffsets", "earliest").load()
          .withColumn("v", col("value").cast("string")))
    return [r.v for r in df.filter(col("v").contains(f'"sensorId": "{sensor}"') & col("v").contains(f'"timestamp": {ts}')).collect()]

def publicar(key, valor_json, veces=1):
    df = spark.createDataFrame([(key, valor_json)] * veces, ["key", "value"])
    df.write.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP).option("topic", TOPIC).save()

def publicar_tardio(q, t0):
    time.sleep(35)
    real = leer_lectura_real("nsmj-ethernet", 30)
    original = json.loads(valor_kafka("nsmj-ethernet", real.timestamp)[0])
    tardio = dict(original, origen="reenvio-tardio")
    publicar("nsmj-ethernet", json.dumps(tardio))
    registro_tardio.update(tardio=tardio, publicado_ms=int(time.time() * 1000), offset_original=(real.partition, real.offset))
    print(f">>> {hora(registro_tardio['publicado_ms'])} publicada de nuevo la lectura real de nsmj-ethernet medida a las "
          f"{hora(tardio['timestamp'])} ({(registro_tardio['publicado_ms'] - tardio['timestamp']) / 1000:.1f} s tarde)")

agregado_tardio = (lecturas
    .withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId"))
    .agg(count("*").alias("n"), sround(avg("bytesRxPorSeg") / 1000, 1).alias("rx_prom_KBs")))

lotes_t, prog_t = correr(agregado_tardio, "tardio", 70, chk=CHK_TARDIO, mientras=publicar_tardio)
lotes_t = [(b, ventana_txt(p) if not p.empty else p) for b, p in lotes_t]


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


...

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [9]:

t = registro_tardio["tardio"]
pub = registro_tardio["publicado_ms"]
v_tardia = ventana_de(t["timestamp"])
print("Evento tardío publicado (lectura real, tal como quedó en Kafka):")
for v in valor_kafka("nsmj-ethernet", t["timestamp"]):
    print("  ", v)
print()
print(f"Medido a las {hora(t['timestamp'])} -> ventana {v_tardia}; publicado a las {hora(pub)} "
      f"({(pub - t['timestamp']) / 1000:.1f} s tarde, margen del watermark = 10 s)")
print()

wm = marca_de_agua(prog_t)
inicio_batch = {p["batchId"]: datetime.fromisoformat(p["timestamp"].replace("Z", "+00:00")).timestamp() * 1000 for p in prog_t}
for b, p in lotes_t:
    if p.empty:
        continue
    for _, f in p[(p["sensorId"] == "nsmj-ethernet") & (p["ventana"] == v_tardia)].iterrows():
        print(f"Batch {b} ({hora(inicio_batch[b])}): ventana {v_tardia} nsmj-ethernet n={f['n']}  watermark {wm_lima(wm.get(b))}")

despues = [b for b, ms in inicio_batch.items() if ms > pub]
reaparece = [b for b, p in lotes_t if b in despues and not p.empty
             and ((p["sensorId"] == "nsmj-ethernet") & (p["ventana"] == v_tardia)).any()]
descartadas = {p["batchId"]: p["stateOperators"][0].get("numRowsDroppedByWatermark", 0)
               for p in prog_t if p["batchId"] in despues and p["stateOperators"]}
print()
print(f"Micro-lotes después de la publicación: {despues}")
print(f"La ventana {v_tardia} reaparece en: {reaparece or 'NINGUNO -> el evento tardío fue descartado'}")
print("numRowsDroppedByWatermark en esos micro-lotes:", {b: n for b, n in descartadas.items() if n} or descartadas)


Evento tardío publicado (lectura real, tal como quedó en Kafka):


   {"tipoEvento": "red.lectura", "sensorId": "nsmj-ethernet", "host": "nsmj", "interfaz": "Ethernet", "tipoInterfaz": "fisica", "velocidadEnlaceMbps": 100, "intervaloMs": 1093, "bytesRxPorSeg": 290.9, "bytesTxPorSeg": 270.8, "paquetesRxPorSeg": 4.6, "paquetesTxPorSeg": 4.6, "erroresPorSeg": 0.0, "descartesPorSeg": 0.0, "origen": "agente-red", "timestamp": 1791497359548}
   {"tipoEvento": "red.lectura", "sensorId": "nsmj-ethernet", "host": "nsmj", "interfaz": "Ethernet", "tipoInterfaz": "fisica", "velocidadEnlaceMbps": 100, "intervaloMs": 1093, "bytesRxPorSeg": 290.9, "bytesTxPorSeg": 270.8, "paquetesRxPorSeg": 4.6, "paquetesTxPorSeg": 4.6, "erroresPorSeg": 0.0, "descartesPorSeg": 0.0, "origen": "reenvio-tardio", "timestamp": 1791497359548}

Medido a las 17:09:19 -> ventana 17:09:10-17:09:20; publicado a las 17:09:53 (33.5 s tarde, margen del watermark = 10 s)

Batch 1 (17:09:20): ventana 17:09:10-17:09:20 nsmj-ethernet n=4  watermark sin referencia (arranque)

Micro-lotes después de la


### 5.1 El arranque en frío: el mismo tipo de dato tardío sí pasa si es lo primero que ve la consulta

Consulta nueva (checkpoint vacío): a los 2 s de arrancar, antes de que el watermark tenga referencia, se vuelve a publicar
una lectura real de `nsmj-ethernet` de hace ~60 s.


In [10]:

CHK_FRIO = f"{BASE}/chk_frio"
limpiar(CHK_FRIO)
registro_frio = {}

def publicar_en_frio(q, t0):
    time.sleep(2)
    real = leer_lectura_real("nsmj-ethernet", 60)
    tardio = dict(json.loads(valor_kafka("nsmj-ethernet", real.timestamp)[0]), origen="reenvio-tardio")
    publicar("nsmj-ethernet", json.dumps(tardio))
    registro_frio.update(tardio=tardio, publicado_ms=int(time.time() * 1000))
    print(f">>> arranque en frío: publicada a las {hora(registro_frio['publicado_ms'])} la lectura medida a las {hora(tardio['timestamp'])}")

lotes_f, prog_f = correr(agregado_tardio, "frio", 20, chk=CHK_FRIO, mientras=publicar_en_frio)
lotes_f = [(b, ventana_txt(p) if not p.empty else p) for b, p in lotes_f]
tf = registro_frio["tardio"]["timestamp"]
v_frio = ventana_de(tf)
wm_f = marca_de_agua(prog_f)
print(f"Lectura de las {hora(tf)} (ventana {v_frio}), publicada {(registro_frio['publicado_ms'] - tf) / 1000:.1f} s tarde")
for b, p in lotes_f:
    if p.empty:
        continue
    for _, f in p[(p["sensorId"] == "nsmj-ethernet") & (p["ventana"] == v_frio)].iterrows():
        print(f"Batch {b} (watermark {wm_lima(wm_f.get(b))}): ACEPTADA, ventana {f['ventana']} n={f['n']}")


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


>>> arranque en frío: publicada a las 17:10:29 la lectura medida a las 17:09:27


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Lectura de las 17:09:27 (ventana 17:09:20-17:09:30), publicada 61.9 s tarde
Batch 1 (watermark sin referencia (arranque)): ACEPTADA, ventana 17:09:20-17:09:30 n=1



## 6. Duplicado real filtrado por `dropDuplicates` acotado por watermark

La lectura real más reciente de `nsmj-ethernet` se vuelve a publicar **dos veces, sin cambiar nada** (lo que pasaría con
un reintento de MQTT QoS 1 o de Kafka *at-least-once*). En Kafka queda tres veces: la original y dos copias.

La clave incluye `ts`, la columna del watermark: así Spark puede olvidar las claves viejas. Con `timestamp` (Long, sin
watermark) el estado de deduplicación nunca se limpiaría (se compara en la celda siguiente).


In [11]:

CHK_DEDUP = f"{BASE}/chk_dedup"
limpiar(CHK_DEDUP)
registro_dup = {}

def publicar_duplicado(q, t0):
    time.sleep(12)
    real = leer_lectura_real("nsmj-ethernet", 3)
    original = valor_kafka("nsmj-ethernet", real.timestamp)[0]
    publicar("nsmj-ethernet", original, veces=2)
    registro_dup.update(ts=real.timestamp, valor=original, publicado_ms=int(time.time() * 1000))
    print(f">>> {hora(registro_dup['publicado_ms'])} lectura real de nsmj-ethernet ({hora(real.timestamp)}) publicada DOS veces más")

deduplicado = (lecturas
    .withWatermark("ts", "30 seconds")
    .dropDuplicates(["sensorId", "ts"]))

lotes_d, prog_d = correr(deduplicado, "dedup", 35, modo="append", trigger="3 seconds", chk=CHK_DEDUP, mientras=publicar_duplicado)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


...

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Exception in thread "stream execution thread for dedup [id = 0db6fab6-2113-4578-bd29-7f2424c90c24, runId = db11382b-7edb-433a-9a77-5740a4e69226]" java.lang.StackOverflowError
	at java.base/java.lang.Character.codePointAt(Character.java:9321)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4106)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern

In [12]:

ts_dup = registro_dup["ts"]
en_kafka = valor_kafka("nsmj-ethernet", ts_dup)
print(f"Veces que la lectura (sensorId=nsmj-ethernet, timestamp={ts_dup}) está en Kafka: {len(en_kafka)}")
salida = [(b, len(p[(p["sensorId"] == "nsmj-ethernet") & (p["timestamp"] == ts_dup)])) for b, p in lotes_d if not p.empty]
print("Veces que sale en la salida deduplicada:", sum(n for _, n in salida), "->", [s for s in salida if s[1]])
total_entrada = sum(p["numInputRows"] for p in prog_d)
total_salida = sum(len(p) for _, p in lotes_d)
print(f"Filas que entraron al stream: {total_entrada}; filas emitidas tras deduplicar: {total_salida}")


Veces que la lectura (sensorId=nsmj-ethernet, timestamp=1791497454268) está en Kafka: 3
Veces que sale en la salida deduplicada: 1 -> [(4, 1)]
Filas que entraron al stream: 126; filas emitidas tras deduplicar: 124


### 6.1 Por qué la clave debe incluir la columna del watermark: estado de `dropDuplicates` en 60 s

In [13]:

def estado_dedup(claves, nombre):
    chk = f"{BASE}/chk_estado_{nombre}"
    limpiar(chk)
    df = lecturas.withWatermark("ts", "10 seconds").dropDuplicates(claves)
    _, prog = correr(df, nombre, 60, modo="append", trigger="5 seconds", chk=chk)
    return [(p["batchId"], p["stateOperators"][0]["numRowsTotal"]) for p in prog if p["stateOperators"]]

con_ts = estado_dedup(["sensorId", "ts"], "con_ts")
con_long = estado_dedup(["sensorId", "timestamp"], "con_long")
print("filas en el estado, clave (sensorId, ts)        :", con_ts)
print("filas en el estado, clave (sensorId, timestamp) :", con_long)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


...

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


filas en el estado, clave (sensorId, ts)        : [(0, 0), (1, 8), (2, 28), (3, 48), (4, 60), (5, 60), (6, 60), (7, 60), (8, 56), (9, 60), (10, 60), (11, 60), (12, 60)]
filas en el estado, clave (sensorId, timestamp) : [(0, 0), (1, 12), (2, 32), (3, 48), (4, 68), (5, 88), (6, 108), (7, 128), (8, 148), (9, 168), (10, 188), (11, 204), (12, 224)]


## 7. Intervalo de disparo: 1 s frente a 5 s, 20 s de reloj cada uno

In [14]:

def contar_microlotes(intervalo, segundos_totales):
    q = (lecturas.writeStream.outputMode("append").format("memory")
         .queryName(f"conteo_{intervalo.replace(' ', '_')}")
         .trigger(processingTime=intervalo).start())
    q.awaitTermination(segundos_totales)
    progreso = q.recentProgress
    q.stop()
    filas = [p["numInputRows"] for p in progreso]
    return progreso[-1]["batchId"], len(progreso), sum(filas), round(sum(filas) / max(len(filas), 1), 1)

for intervalo in ["1 second", "5 seconds"]:
    batch_id, lotes_n, filas, por_lote = contar_microlotes(intervalo, 20)
    print(f"trigger {intervalo:>9}: último batchId={batch_id}  micro-lotes={lotes_n}  filas={filas}  filas/lote={por_lote}")


26/10/08 22:13:42 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 19, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@59e078d0] is aborting.
26/10/08 22:13:42 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 19, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@59e078d0] aborted.
26/10/08 22:13:42 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException: Stage cancelled: [SPARK_JOB_CANCELLED] Job 104 cancelled Query conteo_1_second [id = 0615484d-9b32-4a72-999b-1552c99a9707, runId = 706ec995-39c1-4805-b32c-5147ed6b5a46] was stopped SQLSTATE: XXKDA
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:342)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:610)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at scala.collec

trigger  1 second: último batchId=18  micro-lotes=19  filas=72  filas/lote=3.8


trigger 5 seconds: último batchId=4  micro-lotes=5  filas=68  filas/lote=13.6


## 8. Persistencia en Parquet con checkpoint: dos corridas que acumula

In [15]:

PARQUET = f"{BASE}/red_telemetria_parquet"
CHK_PARQUET = f"{BASE}/chk_parquet"
limpiar(PARQUET); limpiar(CHK_PARQUET)

def corrida_parquet(segundos):
    q = (lecturas.drop("kafka_ts").writeStream.outputMode("append").format("parquet")
         .option("path", PARQUET).option("checkpointLocation", CHK_PARQUET)
         .trigger(processingTime="5 seconds").start())
    q.awaitTermination(segundos)
    q.stop()
    g = spark.read.parquet(PARQUET)
    return g.count(), g.select("partition", "offset").distinct().count(), g

for corrida in (1, 2):
    filas, unicas, guardado = corrida_parquet(30)
    rango = guardado.groupBy("partition").agg(expr("min(offset) AS desde"), expr("max(offset) AS hasta")).orderBy("partition")
    print(f"Corrida {corrida}: {filas} filas guardadas ({unicas} pares partition/offset distintos)")
    rango.show()
    if corrida == 1:
        time.sleep(10)   # entre corridas el agente sigue publicando: la corrida 2 debe retomar desde el checkpoint


Corrida 1: 104 filas guardadas (104 pares partition/offset distintos)


+---------+-----+-----+
|partition|desde|hasta|
+---------+-----+-----+
|        0| 1734| 1785|
|        1|  862|  887|
|        2|  862|  887|
+---------+-----+-----+



Corrida 2: 260 filas guardadas (260 pares partition/offset distintos)
+---------+-----+-----+
|partition|desde|hasta|
+---------+-----+-----+
|        0| 1734| 1863|
|        1|  862|  926|
|        2|  862|  926|
+---------+-----+-----+



In [16]:
def visibles(ruta):
    return sorted(f for f in os.listdir(ruta) if not f.startswith("."))

def por_numero(ruta):
    return sorted((f for f in os.listdir(ruta) if f.split(".")[0].isdigit() and not f.startswith(".")),
                  key=lambda f: int(f.split(".")[0]))

print("Checkpoint de Parquet (sección 8):  ", visibles(CHK_PARQUET))
print("Checkpoint de la ventana (sección 3):", visibles(CHK_VENTANA))
print("  offsets/:", por_numero(f"{CHK_VENTANA}/offsets"))
print("  commits/:", por_numero(f"{CHK_VENTANA}/commits"))
for particion in visibles(f"{CHK_VENTANA}/state/0"):
    print(f"  state/0/{particion}/:", por_numero(f"{CHK_VENTANA}/state/0/{particion}") or visibles(f"{CHK_VENTANA}/state/0/{particion}"))

ultimo = por_numero(f"{CHK_VENTANA}/offsets")[-1]
lineas = open(f"{CHK_VENTANA}/offsets/{ultimo}").read().splitlines()
meta = json.loads(lineas[1])
print(f"\noffsets/{ultimo}  (último micro-lote):")
print("  batchWatermarkMs :", meta["batchWatermarkMs"], "->", hora(meta["batchWatermarkMs"]))
print("  batchTimestampMs :", meta["batchTimestampMs"], "->", hora(meta["batchTimestampMs"]))
print("  offsets de Kafka :", lineas[2])

Checkpoint de Parquet (sección 8):   ['commits', 'metadata', 'offsets', 'sources']
Checkpoint de la ventana (sección 3): ['commits', 'metadata', 'offsets', 'sources', 'state']
  offsets/: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
  commits/: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
  state/0/0/: ['1.delta', '2.delta', '3.delta', '4.delta', '5.delta', '6.delta', '7.delta', '8.delta', '9.delta', '10.delta']
  state/0/1/: ['1.delta', '2.delta', '3.delta', '4.delta', '5.delta', '6.delta', '7.delta', '8.delta', '9.delta', '10.delta']
  state/0/2/: ['1.delta', '2.delta', '3.delta', '4.delta', '5.delta', '6.delta', '7.delta', '8.delta', '9.delta', '10.delta']
  state/0/_metadata/: ['metadata']

offsets/9  (último micro-lote):
  batchWatermarkMs : 1791497339253 -> 17:08:59
  batchTimestampMs : 1791497355008 -> 17:09:15
  offsets de Kafka : {"red-telemetria":{"0":1170,"1":582,"2":582}}
